### Create network for spatial_queue simulation from OSMnx

In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.wkt import loads
from shapely import wkt
import osmnx as ox
import networkx as nx

In [23]:
# create network using bounding box with osmnx
G = ox.graph_from_bbox(37.878, 37.84, -122.267, -122.182, network_type='drive_service') # Caldecott Tunnel area

# convert to geopandas
ox_nodes, ox_edges = ox.graph_to_gdfs(G)
print('Download OSMnx road network, containing {} nodes, {} edges'.format(ox_nodes.shape[0], ox_edges.shape[0]))

Download OSMnx road network, containing 1899 nodes, 4435 edges


In [24]:
# copy the nodes dataset
nodes = ox_nodes.copy().reset_index()
nodes['lon'] = nodes['x']
nodes['lat'] = nodes['y']

# reindex the nid and make a dictionary
nodes['node_id'] = np.arange(nodes.shape[0])
osmid_to_nid_dict = {getattr(i, 'osmid'): getattr(i, 'node_id') for i in nodes.itertuples()}

# clean the node dataset
nodes = nodes[['node_id', 'lon', 'lat', 'geometry']]

nodes.head()

,node_id,lon,lat,geometry
0,0,-122.246530,37.866153,POINT (-122.24653 37.86615)
1,1,-122.247001,37.866140,POINT (-122.24700 37.86614)
2,2,-122.243368,37.860976,POINT (-122.24337 37.86098)
3,3,-122.241571,37.861797,POINT (-122.24157 37.86180)
4,4,-122.230830,37.866458,POINT (-122.23083 37.86646)


In [25]:
# copy the edges dataset
edges = ox_edges.copy().reset_index()

# reindex the columns and fill up missing values
edges['uniqueid'] = np.arange(edges.shape[0])
edges['start_nid'] = edges['v'].map(osmid_to_nid_dict)
edges['end_nid'] = edges['u'].map(osmid_to_nid_dict)
edges['lanes'] = edges['lanes'].fillna('1')
edges['type'] = edges['highway']

# Function to return the first element if it's a list, otherwise return the value
def keep_first_element(value):
    if isinstance(value, list) and len(value) > 0:
        return value[0]
    return value

# Apply to the columns
edges['osmid'] = edges['osmid'].apply(keep_first_element).astype('Int64')
edges['lanes'] = edges['lanes'].apply(keep_first_element).astype('Int64')

# Assume the capacity of different roads according to “A Policy on Geometric Design of Highways and Streets” (the Green Book) by AASHTO
edges['capacity'] = 900 * edges['lanes'] # 900 for urban street with traffic control
edges['capacity'] = np.where(edges['type'] == 'motorway', edges['capacity']*2 , edges['capacity']) # 1800 for highway road
edges['capacity'] = np.where(edges['type'].isin['secondary', 'residential', 'tertiary'], edges['capacity']*0.667 , edges['capacity']) # 600 for mountain road

# Extract the number from the maxspeed column
edges['maxspeed'] = edges['maxspeed'].str.extract(r'(\d+)').astype('Int64')
edges['maxspeed'] = edges['maxspeed'].fillna(30)

# Compute the fft for each link
edges['fft'] = edges['length']/edges['maxspeed']*2.23694

# clean up the edges dataframe
edges = edges[['uniqueid', 'start_nid', 'end_nid', 'oneway', 'osmid', 'length', 'type', 'lanes', 'maxspeed', 'capacity', 'fft', 'geometry']]

edges.head()

,uniqueid,start_nid,end_nid,oneway,osmid,length,type,lanes,maxspeed,capacity,fft,geometry
0,0,1,0,False,188703407,41.398,residential,1,25,900,3.704194,"LINESTRING (-122.24653 37.86615, -122.24700 37..."
1,1,0,1,False,188703407,41.398,residential,1,25,900,3.704194,"LINESTRING (-122.24700 37.86614, -122.24653 37..."
2,2,82,1,False,188703408,57.977,residential,1,25,900,5.187643,"LINESTRING (-122.24700 37.86614, -122.24766 37..."
3,3,1166,1,False,5077756,80.056,service,1,30,900,5.969349,"LINESTRING (-122.24700 37.86614, -122.24699 37..."
4,4,1360,2,False,5149903,107.320,secondary,2,25,1200.6,9.602736,"LINESTRING (-122.24337 37.86098, -122.24328 37..."


In [26]:
# Specify the data type
edges['osmid'] = edges['osmid'].astype('int64')
edges['lanes'] = edges['lanes'].astype('int64')
edges['maxspeed'] = edges['maxspeed'].astype('int64')
edges['capacity'] = edges['capacity'].astype('int64')

In [27]:
# Flip the start_nid and end_nid of oneway roads
edges.loc[edges['oneway'] == True, ['start_nid', 'end_nid']] = edges.loc[edges['oneway'] == True, ['end_nid', 'start_nid']].values

In [28]:
nodes.to_csv('nodes.csv', index = False)
edges.to_csv('edges.csv', index = False)

In [4]:
nodes = pd.read_csv('nodes.csv')
tunnel = pd.read_csv('tunnel.csv')
claremont = pd.read_csv('claremont.csv')
mountain = pd.read_csv('mountain.csv')

In [48]:
updated_edges = tunnel.append(claremont)
updated_edges = updated_edges.append(mountain)
updated_edges = updated_edges.drop_duplicates()

/var/folders/p5/yl_qhkks45xd154yl6qd7th40000gn/T/ipykernel_78405/1985498467.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  updated_edges = tunnel.append(claremont)
/var/folders/p5/yl_qhkks45xd154yl6qd7th40000gn/T/ipykernel_78405/1985498467.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  updated_edges = updated_edges.append(mountain)


In [50]:
updated_nodes = nodes[nodes['node_id'].isin(updated_edges['start_nid'])|nodes['node_id'].isin(updated_edges['end_nid'])]

,node_id,lon,lat,geometry
2,2,-122.243368,37.860976,POINT (-122.2433676 37.8609763)
4,4,-122.230830,37.866458,POINT (-122.2308297 37.8664576)
5,5,-122.217862,37.868358,POINT (-122.2178621 37.8683582)
6,6,-122.244383,37.859827,POINT (-122.2443826 37.8598274)
7,7,-122.244055,37.860731,POINT (-122.2440549 37.860731)
...,...,...,...,...
1682,1682,-122.245569,37.857761,POINT (-122.2455692 37.8577607)
1685,1685,-122.220021,37.854460,POINT (-122.2200212 37.8544597)
1730,1730,-122.224934,37.858064,POINT (-122.2249335 37.8580635)
1759,1759,-122.238167,37.862140,POINT (-122.2381673 37.8621404)


In [52]:
updated_nodes.to_csv('updated_nodes.csv')
updated_edges.to_csv('updated_edges.csv')

In [5]:
tunnel['fft'].sum()

453.5078501735802

In [7]:
claremont['fft'].sum()

646.2146631818684

In [9]:
mountain['fft'].sum()

719.3818373909626